# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Haryomhidhe/Machine-learning-flyrank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

ML-08 Section 1:
Method choice
My lane is refresh and opportunity scoring. The question is which pages should be refreshed first. For this I used Logistic Regression to predict whether a page's clicks will fall in the next month, and I ranked pages by that predicted chance.
The label is 1 if a page's March clicks (scaled by 28/31 to match February's length) fell below 70% of February's while its March impressions stayed at 80% or more of February's. The inputs come only from February 2026, and the label comes only from March 2026, so there is no leakage. I only judged pages with at least 20 clicks in February, because pages with fewer clicks make a very noisy label.
I chose Logistic Regression over a Random Forest or Gradient Boosting for three reasons. First, it is simple and its weights can be read, so I can explain which inputs push a page toward "clicks will fall". Second, the label is noisy and there are only 23 clients, so a more complex model is more likely to learn noise than real patterns. Third, the card says not to reward complexity alone, and a plain model compared honestly against my Week 4 baseline is more useful than a complex one that wins by accident.
The inputs are log impressions, log clicks, average position and CTR, all from February. I used the log of impressions and clicks because those counts are very skewed, and I scaled all four inputs so the weights are comparable.
Limits to state
• Zero-click pages cannot be labelled, so the zero-click picks from my Week 4 rule are not scored here.
• Pages with few clicks make a noisy label, so I set the minimum at 20 February clicks.
• The label measures "got worse", not "is bad".
• 41 pages have an average position below 1, which is impossible for a real position. I kept them and treated them as top ranked.
• Only pages that appear in both February and March can be labelled.

I first considered a label based on the expected CTR for each position, but median CTR did not fall as position improved. It was about 0.16% at positions 1 to 2, 0.25% at 2 to 3, 0.22% at 3 to 5 and 0.16% at 5 to 10, so expected CTR was not a usable reference. I dropped that idea. The two cells with those tables are kept in the notebook as the record of that check.

In [41]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import pandas as pd
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"SELECT DISTINCT regexp_extract(file, 'month=[0-9-]+') AS month FROM glob('{rel}/fact_content_daily_performance/*/*.parquet') ORDER BY month").show()

┌───────────────┐
│     month     │
│    varchar    │
├───────────────┤
│ month=2025-01 │
│ month=2025-02 │
│ month=2025-03 │
│ month=2025-04 │
│ month=2025-05 │
│ month=2025-06 │
│ month=2025-07 │
│ month=2025-08 │
│ month=2025-09 │
│ month=2025-10 │
│ month=2025-11 │
│ month=2025-12 │
│ month=2026-01 │
│ month=2026-02 │
│ month=2026-03 │
│ month=2026-04 │
│ month=2026-05 │
│ month=2026-06 │
├───────────────┤
│    18 rows    │
└───────────────┘



In [42]:
feb = con.sql(f"""
SELECT content_hash_id,
  SUM(gsc_impressions) AS imps,
  SUM(gsc_clicks) AS clicks,
  AVG(gsc_avg_position) AS pos
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-02/*.parquet')
GROUP BY content_hash_id
""").df()

base = feb[feb['imps'] > 100].copy()
base['pos_bucket'] = pd.cut(base['pos'], [0, 2, 3, 5, 10, 20, 50, 10000],
                            labels=['1-2', '2-3', '3-5', '5-10', '10-20', '20-50', '50+'])
expected = base.groupby('pos_bucket', observed=True).agg(
    n=('content_hash_id', 'count'),
    clicks=('clicks', 'sum'),
    imps=('imps', 'sum'))
expected['expected_ctr_pct'] = expected['clicks'] / expected['imps'] * 100
expected

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n,clicks,imps,expected_ctr_pct
pos_bucket,,,,
1-2,4839,41847.0,11866419.0,0.352651
2-3,6235,94953.0,21542278.0,0.440775
3-5,14591,189199.0,51589704.0,0.366738
5-10,26394,154377.0,50888000.0,0.303366
10-20,16706,69201.0,22252028.0,0.310987
20-50,10302,30317.0,19883322.0,0.152475
50+,1056,174.0,366912.0,0.047423


In [43]:
base['ctr'] = base['clicks'] / base['imps'] * 100
base.groupby('pos_bucket', observed=True)['ctr'].agg(
    n='count', median_ctr='median', p75=lambda s: s.quantile(0.75))

,n,median_ctr,p75
pos_bucket,,,
1-2,4839,0.161812,0.376530
2-3,6235,0.248822,0.509431
3-5,14591,0.224803,0.447761
5-10,26394,0.160508,0.402901
10-20,16706,0.073046,0.335219
20-50,10302,0.000000,0.170322
50+,1056,0.000000,0.000000


In [44]:
def month_totals(m):
    return con.sql(f"""
    SELECT content_hash_id,
      SUM(gsc_impressions) AS imps,
      SUM(gsc_clicks) AS clicks
    FROM read_parquet('{rel}/fact_content_daily_performance/month={m}/*.parquet')
    GROUP BY content_hash_id
    """).df()

feb = month_totals('2026-02')
mar = month_totals('2026-03')
d = feb.merge(mar, on='content_hash_id', suffixes=('_feb', '_mar'))
d['clicks_mar_adj'] = d['clicks_mar'] * 28 / 31
d['imps_mar_adj'] = d['imps_mar'] * 28 / 31

for min_clicks in [10, 20, 50]:
    s = d[d['clicks_feb'] >= min_clicks]
    fell = s['clicks_mar_adj'] < 0.7 * s['clicks_feb']
    held = s['imps_mar_adj'] >= 0.8 * s['imps_feb']
    print("min Feb clicks", min_clicks, "| pages:", len(s),
          "| label=1:", round((fell & held).mean() * 100, 1), "%")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

min Feb clicks 10 | pages: 13336 | label=1: 16.1 %
min Feb clicks 20 | pages: 6478 | label=1: 15.1 %
min Feb clicks 50 | pages: 2104 | label=1: 11.8 %


In [45]:
feb_pos = con.sql(f"""
SELECT content_hash_id, AVG(gsc_avg_position) AS pos
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-02/*.parquet')
GROUP BY content_hash_id
""").df()

data = d.merge(feb_pos, on='content_hash_id')
data = data[data['clicks_feb'] >= 20].copy()

data['label'] = ((data['clicks_mar_adj'] < 0.7 * data['clicks_feb']) &
                 (data['imps_mar_adj'] >= 0.8 * data['imps_feb'])).astype(int)

data['ctr_feb'] = data['clicks_feb'] / data['imps_feb'] * 100
severity = 2 * (data['ctr_feb'] < 0.15) + 1 * (data['pos'] > 10)
data['baseline_score'] = (data['imps_feb'] * severity).where(data['imps_feb'] > 100, 0)

print("pages:", len(data))
print("label=1 share:", round(data['label'].mean() * 100, 1), "%")
print("pages where baseline score > 0:", (data['baseline_score'] > 0).sum())
print("pages with pos < 1:", (data['pos'] < 1).sum())

pages: 6478
label=1 share: 15.1 %
pages where baseline score > 0: 1388
pages with pos < 1: 41


In [46]:
bad_ids = data.loc[data['pos'] < 1, ['content_hash_id']]
con.register('bad_ids', bad_ids)

print(con.sql(f"""
SELECT COUNT(*) AS days,
       SUM((gsc_avg_position = 0)::INT) AS zero_days,
       SUM((gsc_avg_position IS NULL)::INT) AS null_days
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-02/*.parquet')
JOIN bad_ids USING (content_hash_id)
""").df())

print(data.loc[data['pos'] < 1, 'pos'].describe())

   days  zero_days  null_days
0  1148        2.0       15.0
count    41.000000
mean      0.761375
std       0.153168
min       0.314707
25%       0.643797
50%       0.783187
75%       0.868165
max       0.977644
Name: pos, dtype: float64


In [47]:
print(con.sql(f"""
DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-02/*.parquet')
""").df()[['column_name', 'column_type']])

print(con.sql(f"""
SELECT DISTINCT regexp_extract(file, 'internship-warehouse/([^/]+)/', 1) AS table_folder
FROM glob('{rel}/**/*.parquet')
""").df())

                 column_name column_type
0                report_date        DATE
1             client_hash_id     VARCHAR
2            content_hash_id     VARCHAR
3             client_has_gsc     BOOLEAN
4             client_has_ga4     BOOLEAN
5         gsc_data_available     BOOLEAN
6         ga4_data_available     BOOLEAN
7            gsc_impressions      BIGINT
8                 gsc_clicks      BIGINT
9           gsc_sum_position      BIGINT
10          gsc_avg_position      DOUBLE
11             ga4_pageviews      BIGINT
12              ga4_sessions      BIGINT
13                 ga4_users      BIGINT
14      ga4_engaged_sessions      BIGINT
15  ga4_total_engagement_sec      BIGINT
16          sessions_organic      BIGINT
17           sessions_direct      BIGINT
18         sessions_referral      BIGINT
19           sessions_social      BIGINT
20             sessions_paid      BIGINT
21               sessions_ai      BIGINT
22                ai_chatgpt      BIGINT
23             a

In [48]:
clients = con.sql(f"""
SELECT content_hash_id,
       COUNT(DISTINCT client_hash_id) AS n_clients,
       MIN(client_hash_id) AS client_hash_id
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-02/*.parquet')
GROUP BY content_hash_id
""").df()

data = data.merge(clients, on='content_hash_id')

print("pages:", len(data))
print("pages linked to more than one client:", (data['n_clients'] > 1).sum())
print("number of clients:", data['client_hash_id'].nunique())
print(data['client_hash_id'].value_counts().head(5))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

pages: 6478
pages linked to more than one client: 0
number of clients: 23
client_hash_id
client_73cda7b4e4f265ea    2003
client_62f4a7e64f5e0096    1432
client_23a62021009f63c4    1321
client_e547b89c05043229     641
client_fef1a8f436438636     395
Name: count, dtype: int64


In [49]:
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)
data = data.reset_index(drop=True)
data['fold'] = -1

for k, (train_idx, test_idx) in enumerate(gkf.split(data, data['label'], groups=data['client_hash_id'])):
    data.loc[test_idx, 'fold'] = k

print(data.groupby('fold').agg(
    pages=('label', 'size'),
    clients=('client_hash_id', 'nunique'),
    label_rate=('label', 'mean')
).round(3))

      pages  clients  label_rate
fold                            
0      2003        1       0.144
1      1432        1       0.188
2      1321        1       0.153
3       861        4       0.111
4       861       16       0.141


In [50]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

data['log_imps'] = np.log1p(data['imps_feb'])
data['log_clicks'] = np.log1p(data['clicks_feb'])
features = ['log_imps', 'log_clicks', 'pos', 'ctr_feb']

def prec_at_k(y, score, k=50, seed=0):
    rng = np.random.default_rng(seed)
    order = np.lexsort((rng.random(len(score)), -score))
    return y[order[:k]].mean()

rows = []
for k in range(5):
    tr = data[data['fold'] != k]
    te = data[data['fold'] == k]
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    model.fit(tr[features], tr['label'])
    p = model.predict_proba(te[features])[:, 1]
    y = te['label'].values
    rows.append({
        'fold': k,
        'base_rate': y.mean(),
        'model': prec_at_k(y, p),
        'baseline': prec_at_k(y, te['baseline_score'].values),
        'random': np.mean([prec_at_k(y, np.random.default_rng(s).random(len(te)), seed=s) for s in range(200)])
    })

res = pd.DataFrame(rows).round(3)
print(res)
print(res[['model', 'baseline', 'random']].mean().round(3))

   fold  base_rate  model  baseline  random
0     0      0.144   0.20      0.08   0.147
1     1      0.188   0.42      0.06   0.185
2     2      0.153   0.16      0.16   0.156
3     3      0.111   0.16      0.10   0.112
4     4      0.141   0.10      0.10   0.135
model       0.208
baseline    0.100
random      0.147
dtype: float64


In [51]:
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(data[features], data['label'])
coefs = pd.Series(model[-1].coef_[0], index=features).round(3)
print(coefs.sort_values())

log_imps     -0.301
ctr_feb      -0.289
log_clicks    0.035
pos           0.046
dtype: float64


In [52]:
rows = []
for k in range(5):
    tr = data[data['fold'] != k]
    te = data[(data['fold'] == k) & (data['clicks_feb'] >= 50)]
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    model.fit(tr[features], tr['label'])
    p = model.predict_proba(te[features])[:, 1]
    y = te['label'].values
    rows.append({'fold': k, 'pages': len(te), 'base_rate': y.mean(),
                 'model': prec_at_k(y, p), 'baseline': prec_at_k(y, te['baseline_score'].values)})
print(pd.DataFrame(rows).round(3))

   fold  pages  base_rate  model  baseline
0     0    705      0.104   0.08      0.08
1     1    484      0.151   0.16      0.08
2     2    393      0.117   0.12      0.08
3     3    291      0.096   0.14      0.14
4     4    231      0.126   0.16      0.10


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Every page in the data belongs to exactly one client, and there are 23 clients. Pages from the same client tend to behave alike, so if some of a client's pages sit in training and others in testing, the model can quietly learn that client's habits and look better than it really is. To avoid this I used 5 fold grouped cross validation, with the client as the group. Each client is held out whole, exactly once, so the model is always tested on clients it has never seen.

The clients are very uneven in size. The three largest hold 2,003, 1,432 and 1,321 pages, and each ended up alone as its own test fold. The fourth fold holds 4 mid sized clients and the fifth holds the 16 smallest. The label rate across the test folds ranges from 11.1% to 18.8%.

The model, my Week 4 baseline rule and random guessing are all scored on the same pages in the same test folds, using the same metric. Inputs come only from February 2026 and the label comes only from March 2026, so nothing from the future leaks into the inputs.

The metric is precision at the top 50. For each test fold I take the 50 pages ranked highest and count what share of them really had their clicks fall. I did not use accuracy, because about 85% of pages are label 0, so a model that always says 0 would score about 85% while finding nothing. Ties in the baseline score are broken at random so the baseline is not treated unfairly.

In [53]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feb_pos = con.sql(f"""
SELECT content_hash_id, AVG(gsc_avg_position) AS pos
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-02/*.parquet')
GROUP BY content_hash_id
""").df()

data = d.merge(feb_pos, on='content_hash_id')
data = data[data['clicks_feb'] >= 20].copy()

data['label'] = ((data['clicks_mar_adj'] < 0.7 * data['clicks_feb']) &
                 (data['imps_mar_adj'] >= 0.8 * data['imps_feb'])).astype(int)

data['ctr_feb'] = data['clicks_feb'] / data['imps_feb'] * 100
severity = 2 * (data['ctr_feb'] < 0.15) + 1 * (data['pos'] > 10)
data['baseline_score'] = (data['imps_feb'] * severity).where(data['imps_feb'] > 100, 0)

print("pages:", len(data))
print("label=1 share:", round(data['label'].mean() * 100, 1), "%")
print("pages where baseline score > 0:", (data['baseline_score'] > 0).sum())

pages: 6478
label=1 share: 15.1 %
pages where baseline score > 0: 1388


In [54]:
clients = con.sql(f"""
SELECT content_hash_id,
       COUNT(DISTINCT client_hash_id) AS n_clients,
       MIN(client_hash_id) AS client_hash_id
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-02/*.parquet')
GROUP BY content_hash_id
""").df()

data = data.merge(clients, on='content_hash_id')

print("pages:", len(data))
print("pages linked to more than one client:", (data['n_clients'] > 1).sum())
print("number of clients:", data['client_hash_id'].nunique())

pages: 6478
pages linked to more than one client: 0
number of clients: 23


In [55]:
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)
data = data.reset_index(drop=True)
data['fold'] = -1

for k, (train_idx, test_idx) in enumerate(gkf.split(data, data['label'], groups=data['client_hash_id'])):
    data.loc[test_idx, 'fold'] = k

print(data.groupby('fold').agg(
    pages=('label', 'size'),
    clients=('client_hash_id', 'nunique'),
    label_rate=('label', 'mean')
).round(3))

      pages  clients  label_rate
fold                            
0      2003        1       0.144
1      1432        1       0.188
2      1321        1       0.153
3       861        4       0.111
4       861       16       0.141


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

For each fold, a Logistic Regression was trained on the other four folds and scored on the held out fold. The baseline is my Week 4 rule applied to the same February numbers, and random guessing is the average of 200 random rankings.

| Test fold | Pages | Base rate | Model | Baseline | Random |
|---|---|---|---|---|---|
| 0 | 2,003 | 14.4% | 20% | 8% | 14.7% |
| 1 | 1,432 | 18.8% | 42% | 6% | 18.5% |
| 2 | 1,321 | 15.3% | 16% | 16% | 15.6% |
| 3 | 861 | 11.1% | 16% | 10% | 11.2% |
| 4 | 861 | 14.1% | 10% | 10% | 13.5% |
| **Average** | | | **20.8%** | **10.0%** | **14.7%** |

On this full set the model beat both the baseline and random guessing on average. But the result rests mostly on one fold. Without fold 1 the model averages 15.5%, against 13.8% for random and 11.0% for the baseline. Each fold score is based on only 50 pages, so one page is worth 2 percentage points and small gaps are within noise.

I then repeated the test using only pages with at least 50 February clicks, to remove the noisiest labels.

| Test fold | Pages | Base rate | Model | Baseline |
|---|---|---|---|---|
| 0 | 705 | 10.4% | 8% | 8% |
| 1 | 484 | 15.1% | 16% | 8% |
| 2 | 393 | 11.7% | 12% | 8% |
| 3 | 291 | 9.6% | 14% | 14% |
| 4 | 231 | 12.6% | 16% | 10% |
| **Average** | | **11.9%** | **13.2%** | **9.6%** |

Here the model is close to the base rate, so most of its earlier edge disappears. The baseline scores zero for most pages, so its top 50 is partly filled by random tie breaking, and its numbers on these small test sets shift a little between runs.

In [56]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

data['log_imps'] = np.log1p(data['imps_feb'])
data['log_clicks'] = np.log1p(data['clicks_feb'])
features = ['log_imps', 'log_clicks', 'pos', 'ctr_feb']

def prec_at_k(y, score, k=50, seed=0):
    rng = np.random.default_rng(seed)
    order = np.lexsort((rng.random(len(score)), -score))
    return y[order[:k]].mean()

rows = []
for k in range(5):
    tr = data[data['fold'] != k]
    te = data[data['fold'] == k]
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    model.fit(tr[features], tr['label'])
    p = model.predict_proba(te[features])[:, 1]
    y = te['label'].values
    rows.append({
        'fold': k,
        'base_rate': y.mean(),
        'model': prec_at_k(y, p),
        'baseline': prec_at_k(y, te['baseline_score'].values),
        'random': np.mean([prec_at_k(y, np.random.default_rng(s).random(len(te)), seed=s) for s in range(200)])
    })

res = pd.DataFrame(rows).round(3)
print(res)
print(res[['model', 'baseline', 'random']].mean().round(3))

   fold  base_rate  model  baseline  random
0     0      0.144   0.20      0.08   0.147
1     1      0.188   0.42      0.06   0.185
2     2      0.153   0.16      0.16   0.156
3     3      0.111   0.16      0.10   0.112
4     4      0.141   0.10      0.10   0.135
model       0.208
baseline    0.100
random      0.147
dtype: float64


In [57]:
rows = []
for k in range(5):
    tr = data[data['fold'] != k]
    te = data[(data['fold'] == k) & (data['clicks_feb'] >= 50)]
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    model.fit(tr[features], tr['label'])
    p = model.predict_proba(te[features])[:, 1]
    y = te['label'].values
    rows.append({'fold': k, 'pages': len(te), 'base_rate': y.mean(),
                 'model': prec_at_k(y, p), 'baseline': prec_at_k(y, te['baseline_score'].values)})
print(pd.DataFrame(rows).round(3))

   fold  pages  base_rate  model  baseline
0     0    705      0.104   0.08      0.08
1     1    484      0.151   0.16      0.08
2     2    393      0.117   0.12      0.08
3     3    291      0.096   0.14      0.14
4     4    231      0.126   0.16      0.10


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The weights of the Logistic Regression, fitted once on all pages only to read them, were: log impressions -0.30, CTR -0.29, log clicks 0.04 and average position 0.05. Impressions, clicks and CTR are linked by arithmetic, since CTR is clicks divided by impressions, so I do not read these weights one by one. What they show is that the model leans on impressions and CTR and barely uses average position.

To see what the errors look like, I took the model's 50 highest ranked pages in each test fold, 250 pages in total. 79.2% of them were wrong, which matches the precision of about 21%. The median right pick and the median wrong pick look almost the same, at about 3,000 impressions, 21 clicks and a CTR of 0.7%. Only average position differs, at 5.7 for right picks and 6.5 for wrong ones, which is a small gap. So these inputs do not separate the pages that really fell from the ones that did not.

The median of 21 clicks matters too. The minimum was 20, so the top picks sit right at the edge of the noisiest pages, where a drop below 70% can happen by chance. Together with the 50 plus clicks result in section 3, this suggests the model's edge comes mostly from noisy low click pages, not from spotting real decline.

The baseline showed no skill on this label. It scored 10.0% against 14.7% for random guessing, but with only 250 picks that gap could be chance, so I cannot say it did worse than random. Its score is mostly driven by average position worse than 10, and with 20 or more clicks a page needs over 13,000 impressions to fall below 0.15% CTR, so the CTR part of the rule almost never fires. About 21% of pages get a baseline score above zero and the rest are tied at zero.

One caveat on the comparison. My Week 4 rule was built to find pages with low CTR and poor position, not pages whose clicks fall the following month. The model was trained on the March label and the rule was not. So the table shows that the rule does not predict this label. It does not show that the rule is a bad way to find opportunities.

On fold 4, which holds the 16 smallest clients, the model scored 5 hits out of 50 where about 7 were expected from the base rate. That is a difference of two pages, so I do not draw a conclusion about small clients from it.

Overall, neither the model nor the baseline reliably finds pages whose clicks are really falling using only February inputs. The honest conclusion is directional, not proven.

In [58]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(data[features], data['label'])
coefs = pd.Series(model[-1].coef_[0], index=features).round(3)
print(coefs.sort_values())

log_imps     -0.301
ctr_feb      -0.289
log_clicks    0.035
pos           0.046
dtype: float64


In [59]:
data['model_score'] = np.nan
for k in range(5):
    tr = data[data['fold'] != k]
    te_idx = data.index[data['fold'] == k]
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    m.fit(tr[features], tr['label'])
    data.loc[te_idx, 'model_score'] = m.predict_proba(data.loc[te_idx, features])[:, 1]

top = data.sort_values('model_score', ascending=False).groupby('fold').head(50).copy()
top['wrong'] = top['label'] == 0

print("share of top picks that were wrong:", round(top['wrong'].mean() * 100, 1), "%")
print(top.groupby('wrong')[['imps_feb', 'clicks_feb', 'pos', 'ctr_feb']].median().round(2))

share of top picks that were wrong: 79.2 %
       imps_feb  clicks_feb   pos  ctr_feb
wrong                                     
False    3084.5        21.0  5.72     0.73
True     3063.0        21.0  6.50     0.70


## Self-check

Before you submit, confirm each line honestly:

- ✅ Every section above is filled — markdown thinking AND the code that backs it
- ✅ The notebook runs top to bottom with no errors (Runtime → Run all)
- ✅ No client names, URLs, or private queries anywhere
- ✅ My claims use careful words: observed, measured, directional, decision-support
- ✅ Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

Leakage: inputs come only from February 2026 and the label only from March 2026. In every fold, each client sits entirely in training or entirely in testing, never both.

Same split and same metric: the model, my Week 4 baseline and random guessing are all scored on the same test pages with the same metric, precision at the top 50. Random guessing is the average of 200 random rankings.

Why not accuracy: about 85% of pages are label 0, so accuracy would reward a model that never flags anything.

Method choice: a simple Logistic Regression, chosen over a Random Forest because the label is noisy and there are only 23 clients, so a complex model is more likely to learn noise.

How much to trust the result: one fold carries the full set result, each fold score rests on only 50 pages, and the model's edge mostly disappears on pages with 50 or more clicks. The baseline's numbers on small test sets also shift a little between runs, because its top 50 is partly filled by random tie breaking. I report the result as directional, not proven.

Limits:
- Zero click pages cannot be labelled, so the zero click picks from my Week 4 rule are not scored here.
- Pages with few clicks make a noisy label, so I set the minimum at 20 February clicks.
- The label measures "got worse", not "is bad".
- Only pages that appear in both February and March are included.
- 41 pages have an average position below 1, which is impossible for a real position. I kept them and treated them as top ranked.
- The 70% and 80% label thresholds are starting guesses that I did not tune.